This notebook aim to save and combine data akademik dan data pedoman akademik

### Data Pedoman Akademik (Dense Method)

In [2]:
data_pedoman_dan_rule = {
    "data": [
        "https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/",
        "https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/",
        "https://akademik.nurulfikri.ac.id/2-aturan/",
        "https://akademik.nurulfikri.ac.id/3-kode-etik-mahasiswa/",
        "https://akademik.nurulfikri.ac.id/4-suasana-akademik/",
        "https://akademik.nurulfikri.ac.id/4-profil-dosen/",
        "https://akademik.nurulfikri.ac.id/1-sejarah/",
    ],
    "sumber_data": [
        "satuan kredit semester",
        "syarat kelulusan",
        "aturan dan kode etik",
        "kode etik mahasiswa",
        "suasana akademik",
        "profile dosen",
        "sejarah sttnf",
    ],
}

In [3]:
import os 


LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(data_pedoman_dan_rule["data"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [42]:
pages = []

for doc in loader.lazy_load():
    pages.append(doc)

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_splits = text_splitter.split_documents(pages)


### Model Bahasa (IndoBert)

In [7]:
# memanggil Indobert dari transformer
from transformers import BertTokenizer, AutoModel

tokenizer = BertTokenizer.from_pretrained("Indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/Users/a/Programming/Langchain-Project/my-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# membuat class model embdding
from typing import List 
from langchain_core.embeddings import Embeddings
import torch

class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # polling token menjadi satu vector kalimat
        token_embeddings = outputs.last_hidden_state

        # melakukan mean polling
        sentence_embeddings = token_embeddings.mean(dim=1)

        # konversi ke list python
        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # metode untuk pencarian query pada chroma 
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)


In [9]:
embeddings = IndoBertEmbeddings()

In [10]:
from langchain_elasticsearch import ElasticsearchStore

In [11]:
vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="langchain_index",
    embedding=embeddings,
    es_user="elastic",
    es_password="Xkhwf3uB",
)

In [24]:
vector_store.add_documents(docs_splits)

2026-02-04 16:22:16,990 - INFO - HEAD http://localhost:9200/langchain_index [status:404 duration:0.005s]
2026-02-04 16:22:17,470 - INFO - PUT http://localhost:9200/langchain_index [status:200 duration:0.251s]
2026-02-04 16:22:23,137 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.144s]


['948ff870-0f30-48c7-9d0d-584c39d1a273',
 'c801319d-57e1-428f-87ce-ad7fa3756f42',
 'ed83d94c-a2ae-4581-abfd-36799d245dda',
 'c0ce7ed9-0e84-41c0-a06b-58bfc26995d3',
 '668c4a22-4e1d-41e1-ae85-78565cbdab93',
 'c487dad0-1fc2-460e-b07a-c82a7a648f24',
 'a302fa44-59cf-4713-bcc4-2f54f19670fb',
 '3dd6a756-d0ce-41a8-a5ac-0e5d2f5008a0',
 '227bf5a6-6cd3-4186-b0f8-272f336e666b',
 'fe36105a-c590-4c45-9b27-8867d848764c',
 '14f55e08-9467-4323-b840-6afbac2d930a',
 '90da0f9c-205d-42ee-af00-0ea3364a7622',
 '02c11049-576f-4c47-853a-3b3df0f2fd85',
 '1efafd3f-9dab-4c14-97a0-13ded34c1cdf',
 '1dfe658a-93d3-4ed2-881b-dedc525c35ca',
 '648029f4-1304-4e70-9385-cb4b5b15d56a',
 'acc720e1-5c9d-4dc7-af69-fd02e3212e98',
 'bfed269a-0bd4-484e-b523-6357d2536140',
 'c6129920-7b3d-4a77-9a64-875a187d582a',
 '2a179d76-442b-439c-98a5-2aee69fa1412',
 '148f80db-7d2b-43e8-96ea-a8affe4a6df4',
 '7bd2648f-478d-41f2-8e21-d3e291d049dd',
 '273abfe2-0137-42ec-8775-f52bba8eaf9d',
 'b91cb1a5-b34c-4306-916e-9b123ffbe56f',
 'bd5dad9c-8c57-

### Testing Dense Retriever

In [12]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.2}
)


In [13]:
retriever.invoke("Berapa batas minimum IPK agar mahasiswa dinyatakan lulus?")

[Document(metadata={'source': 'https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/', 'title': '1. Satuan Kredit Semester (SKS) – Pedoman Akademik STT-NF', 'language': 'en-US'}, page_content='Masa Studi\nKetentuan masa studi adalah sebagai berikut:\n\nMasa studi adalah masa untuk penyelesaian beban studi dalam mengikuti proses pendidikan pada program studinya.\nProgram sarjana harus diselesaikan dalam waktu tidak lebih dari tujuh tahun (14 Semester), terhitung mulai saat mahasiswa terdaftar sebagai mahasiswa. Jika ternyata sampai batas masa studi yang ditentukan, mahasiswa belum dapat menyelesaikan studi sarjananya, maka yang bersangkutan dinyatakan tidak mampu melanjutkan studinya/ Drop Out (DO).\nMasa studi tujuh tahun tersebut termasuk cuti akademik dan bagi mahasiswa yang tidak mendaftar ulang per semester tetap diperhitungkan sebagai masa studi.\nBagi mahasiswa yang melampaui masa studi empat tahun (8 Semester) akan diberlakukan ketentuan SPP Progresif.\n\n\n\n\nLast Mo

### Data Akademik Mahasiswa (Sparse Method)

In [14]:
data_akademik = [
    '/Users/a/Programming/Langchain-Project/external-data/sintetik-data-akademik-mahasiswa (2).xlsx'
]

### Processing document

In [15]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter

In [87]:
class DoclingLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [88]:
loader = DoclingLoader(data_akademik)

docs_akademik = loader.lazy_load()

In [89]:
data_akademik_split = text_splitter.split_documents(docs_akademik)

2026-02-05 12:58:33,875 - INFO - detected formats: [<InputFormat.XLSX: 'xlsx'>]
2026-02-05 12:58:33,956 - INFO - Going to convert document batch...
2026-02-05 12:58:33,957 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-02-05 12:58:33,958 - INFO - Processing document sintetik-data-akademik-mahasiswa (2).xlsx
2026-02-05 12:58:33,959 - INFO - Processing sheet 0: Data Mahasiswa Sintetik untuk R
2026-02-05 12:58:33,966 - INFO - Finished converting document sintetik-data-akademik-mahasiswa (2).xlsx in 0.10 sec.


In [16]:
vector_store_sparse = ElasticsearchStore(
    es_url="http://localhost:9200",
    index_name="test_index",
    es_user="elastic",
    es_password="Xkhwf3uB",
    strategy=ElasticsearchStore.BM25RetrievalStrategy(),
)

2026-02-06 14:30:33,092 - INFO - GET http://localhost:9200/ [status:200 duration:0.008s]


In [29]:
vector_store_sparse.add_documents(data_akademik_split)

2026-02-04 16:22:45,578 - INFO - HEAD http://localhost:9200/test_index [status:404 duration:0.003s]
2026-02-04 16:22:45,696 - INFO - PUT http://localhost:9200/test_index [status:200 duration:0.118s]
2026-02-04 16:22:45,727 - INFO - PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.030s]


['a90c9573-09f1-4a7a-9d88-e47600f46e3b',
 '12867448-e31f-4023-a83f-8cf6e5044d82',
 '4b5d04ac-6923-42fb-b640-9ce215e9e66f',
 '5d56f4dd-f242-4d5c-a7e3-8576dde01b7e',
 '265ef853-2ef3-4662-ad3f-026d612551f2',
 'b0e3227d-3077-48bf-b114-241972119a99',
 '7be5edd7-e1e7-4923-8c37-3262c54a2c8d',
 'f5b03288-c029-42aa-b00e-80dc41058277',
 'a29db36d-222d-493d-9260-45af2327037f',
 'f3cacb80-77e6-4c03-8bf2-6858595e1565',
 '7dbf0e00-4014-4e51-bfd0-766e48303b40',
 '22229420-9058-40e2-9d8b-af75326f5468',
 '2f7309be-c2b6-42db-b9ec-a2d9080cfd0b',
 '1e1a636c-74e8-44f1-b7c4-986a0395ac6b',
 'c5efe8db-e078-434f-90ff-69d600d46e4e',
 'a264992e-634d-48c8-8399-a2becd78aade',
 '560a19ab-914a-4a93-af84-c694126f1805']

In [17]:
vector_store_sparse.similarity_search("berapa ipk romi wahyudi")

2026-02-06 14:30:36,022 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.024s]


[Document(metadata={}, page_content='|   17 | 2.021e+07   | Qori Handayani        | T. Informatika   |          5 |          20 |  3.15 |  3.2  |              87 | Aktif     |\n|   18 | 2.021e+07   | Romi Wahyudi Hasibuan | T. Informatika   |          5 |          18 |  2.5  |  2.8  |              75 | Aktif     |\n|   19 | 2.021e+07   | Siti Aminah           | T. Informatika   |          5 |          24 |  4    |  3.95 |              99 | Aktif     |\n|   20 | 2.021e+07   | Tono Suherman         | T. Informatika   |          5 |          15 |  0.5  |  1.8  |              30 | Non-Aktif |\n|   21 | 2.021e+07   | Usman Affandi         | T. Informatika   |          5 |          20 |  3.05 |  3.1  |              85 | Aktif     |\n|   22 | 2.021e+07   | Vina Panduwinata      | T. Informatika   |          5 |          22 |  3.55 |  3.6  |              93 | Aktif     |\n|   23 | 2.021e+07   | Wahyu Hidayat         | T. Informatika   |          5 |          20 |  2.85 |  3    |              8

### Create Agent Which can separate the work to do the conditioning

In [18]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import operator
from langchain_core.messages import AIMessage

In [19]:
class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], operator.add]

### Tools 

In [40]:

@tool
def query_from_academic_rule(query: str):
    """
    Gunakan tool ini HANYA untuk pertanyaan tentang ATURAN, KEBIJAKAN, SYARAT, atau PROSEDUR KAMPUS.
    JANGAN gunakan ini untuk mencari nama orang, nilai, atau data pribadi mahasiswa.
    Contoh input: "syarat yudisium", "aturan cuti", "biaya semester".
    """
    try:
        docs = vector_store.similarity_search(query, k=3)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data pedoman akademik."
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari Pedoman Akademik:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi Kesalahan saat mengakses vector database"


@tool
def get_student_academic_record(query: str):
    """
    Gunakan tool ini untuk mencari DATA PRIBADI MAHASISWA tertentu.
    Termasuk: Nama lengkap, NIM (Nomor Induk Mahasiswa), IPK, Nilai, dan Status.
    Jika user bertanya "Siapa NIM dari Budi?", gunakan tool ini.
    Contoh input: "Budi Santoso", "Romi Wahyudi".
    """
    try:
        docs = vector_store_sparse.similarity_search(query)

        if not docs:
            return "Maaf, tidak ditemukan informasi relevan di data akademik mahasiswa"
        
        formatted_results = "\n\n---\n\n".join([d.page_content for d in docs])
        return f"Ditemukan informasi berikut dari data akademik mahasiswa:\n{formatted_results}"
    
    except Exception as e:
        return "Terjadi kesalahn saat mengakses data akademik"

In [41]:
import json
from langchain_core.messages import AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, END

class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

   
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
    
        if not isinstance(result, AIMessage):
            return False

      
        if len(result.tool_calls) > 0:
            return True
            
        content = result.content if result.content else ""
        if '[{"name":' in content or "tool_calls" in content:
            print("🕵️ Terdeteksi JSON Tool Call di dalam teks!")
            return True
            
        return False

    def call_model(self, state: AgentState):
        messages = state["messages"]
        
        # --- CIRCUIT BREAKER (PENCEGAH LOOP) ---
        # Jika percakapan sudah > 10 langkah (User-AI-Tool-AI-Tool...), hentikan paksa.
        if len(messages) > 10:
             return {'messages': [AIMessage(content="Maaf, saya mencoba mencari tapi prosesnya terlalu lama (Loop detected). Mohon perjelas pertanyaan.")]}

        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        
        try:
            # Tambahkan print debug
            print("🤖 Model sedang berpikir...")
            response = self.model.invoke(messages)
            return {'messages': [response]}
        except Exception as e:
            print(f"Error invoke: {e}")
            return {'messages': [AIMessage(content="Error system.")]}

  
    def take_action(self, state: AgentState):
        last_message = state['messages'][-1]
        results = []
        tool_calls = []

        # SKENARIO A: Native Tool Calls (Model bekerja sempurna)
        if hasattr(last_message, 'tool_calls') and len(last_message.tool_calls) > 0:
            tool_calls = last_message.tool_calls
            
        # SKENARIO B: Manual Parsing dari Teks (Model agak bandel)
        else:
            try:
                content = last_message.content
                # Cari posisi kurung siku JSON [...]
                start_idx = content.find('[{"name":')
                if start_idx != -1:
                    json_str = content[start_idx:]
                    # Bersihkan jika ada sisa text di belakang (opsional)
                    end_idx = json_str.rfind('}]') + 2
                    json_str = json_str[:end_idx]
                    
                    parsed_tools = json.loads(json_str)
                    
                    # Konversi ke format standard tool call
                    for pt in parsed_tools:
                        tool_calls.append({
                            'name': pt['name'],
                            'args': pt['arguments'],
                            'id': 'manual_call' # ID dummy
                        })
            except Exception as e:
                print(f"❌ Gagal parsing manual JSON: {e}")

        # EKSEKUSI TOOLS
        for t in tool_calls:
            print(f"🛠️ Eksekusi Tool: {t['name']} dengan args: {t['args']}")
            
            if t['name'] not in self.tools:
                result = "Error: Tool name not found. Please check valid tools."
            else:
                try:
                    # Pastikan args adalah dict
                    args = t['args']
                    if isinstance(args, str):
                        args = json.loads(args)
                        
                    result = self.tools[t['name']].invoke(args)
                    
                    # --- FIX PENTING: Handle Empty Result ---
                    # Jika hasil kosong, beri tahu model secara eksplisit
                    if not result:
                        result = "Info: Data tidak ditemukan di database untuk input tersebut."
                        
                except Exception as e:
                    result = f"Error execution: {str(e)}"
            
            # Print hasil tool di console biar kamu tau
            print(f"   📄 Hasil Tool: {str(result)[:100]}...") 

            results.append(ToolMessage(
                tool_call_id=t.get('id', 'manual_call'), 
                name=t['name'], 
                content=str(result)
            ))
            
        print("🔙 Kembali ke Model membawa data...")
        return {'messages': results}

### LLM model

In [42]:
from langchain_ollama import ChatOllama

In [43]:
system_prompt = """
Kamu adalah Asisten Akademik Kampus yang cerdas. Tugasmu adalah menjawab pertanyaan user.
Kamu memiliki akses ke dua alat:
1. `lookup_academic_policy`: Untuk mencari aturan umum (Pedoman).
2. `get_student_academic_record`: Untuk mencari data pribadi mahasiswa (Database).

STRATEGI ROUTING:
- Jika user bertanya ATURAN UMUM -> Gunakan `query_from_academic_rule`.
- Jika user bertanya DATA PRIBADI -> Gunakan `get_student_academic_record`.
- Jika user bertanya KEDUANYA (misal: "Apakah saya memenuhi syarat?"), panggil KEDUA alat tersebut.
- Jika user hanya menyapa (Halo/Hi) -> JANGAN panggil alat, jawab langsung dengan sopan.
"""

In [44]:
model = ChatOllama(
    model="mistral:7b-instruct-v0.3-q8_0", 
    temperature=0, 
    streaming=True)

In [46]:
tools = [query_from_academic_rule, get_student_academic_record]

In [47]:
tools[1].name

'get_student_academic_record'

In [48]:
bot = Agent(model, tools, system=system_prompt)

In [51]:
result = bot.graph.invoke({"messages": [HumanMessage(content="bisa bantu carikan berapa ipk romi wahyudi hasibuan")]})

🤖 Model sedang berpikir...


2026-02-06 14:41:50,027 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-06 14:41:59,038 - INFO - POST http://localhost:9200/test_index/_search?_source_includes=metadata,text [status:200 duration:0.028s]


🕵️ Terdeteksi JSON Tool Call di dalam teks!
🛠️ Eksekusi Tool: get_student_academic_record dengan args: {'query': 'Romi Wahyudi'}
   📄 Hasil Tool: Ditemukan informasi berikut dari data akademik mahasiswa:
|   17 | 2.021e+07   | Qori Handayani     ...
🔙 Kembali ke Model membawa data...
🤖 Model sedang berpikir...


2026-02-06 14:42:03,482 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [52]:
print(result['messages'][-1].content)

 Berdasarkan data yang didapatkan, Romi Wahyudi Hasibuan memiliki IPK sebesar 2.8 di semester ke-18.
